In [ ]:
data_dir = "DATA_DIRECTORY"

# For feature selection

In [ ]:
import os
import numpy as np
import pandas as pd

import joblib
from sklearn.preprocessing import MinMaxScaler

In [ ]:
if not os.path.exists(os.path.join(data_dir, "Datasets (feature selection)")): 
    os.mkdir(os.path.join(data_dir, "Datasets (feature selection)"))

In [ ]:
flux = pd.concat([
    pd.read_hdf(os.path.join(data_dir, "peace.h5")),
    pd.read_hdf(os.path.join(data_dir, "rapid.h5"))
    ])
flux.index = pd.to_datetime(flux.index, utc=True)

In [ ]:
pos = pd.read_hdf(os.path.join(data_dir, "position.h5"))
pos.index = pd.to_datetime(pos.index, utc=True)

In [ ]:
drivers = pd.read_hdf(os.path.join(data_dir, "driver_history.h5"))

In [ ]:
tilt = pd.read_hdf(os.path.join(data_dir, "tilts.h5"))
tilt.index = pd.to_datetime(tilt.index, utc=True)

In [ ]:
df = flux.merge(pos, on=["Time tag", "C"]).merge(drivers, on="Time tag").merge(tilt, on="Time tag")

In [ ]:
df = df.resample("10min").mean()
df = df.sample(frac=0.5).sort_values(["Time tag"])

In [ ]:
df["cosMLTcosLAT"] = np.cos(df["MLT"]*np.pi/12) * np.cos(np.radians(df["LAT"]))
df["sinMLTcosLAT"] = np.sin(df["MLT"]*np.pi/12) * np.cos(np.radians(df["LAT"]))
df["sinLAT"] = np.sin(np.radians(df["LAT"]))

In [ ]:
df["lg E"] = np.log10(df["Energy"])

In [ ]:
df = df[[*[f"Flux {a}" for a in np.arange(10, 180, 20)], "lg E", "RAD", "cosMLTcosLAT", "sinMLTcosLAT", "sinLAT", "Dipole tilt", *drivers.columns]]

In [ ]:
df.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets (feature selection)", "dataset.h5"), key="df", format="table")

# For the final model

In [ ]:
import os
import numpy as np
import pandas as pd

import joblib
from sklearn.preprocessing import MinMaxScaler

In [ ]:
if not os.path.exists(os.path.join(data_dir, "Datasets")): 
    os.mkdir(os.path.join(data_dir, "Datasets"))

In [ ]:
flux = pd.concat([
    pd.read_hdf(os.path.join(data_dir, "peace.h5")),
    pd.read_hdf(os.path.join(data_dir, "rapid.h5"))
    ])
flux.index = pd.to_datetime(flux.index, utc=True)

In [ ]:
pos = pd.read_hdf(os.path.join(data_dir, "position.h5"))
pos.index = pd.to_datetime(pos.index, utc=True)

In [ ]:
drivers = pd.read_hdf(os.path.join(data_dir, "driver_history.h5"))

In [ ]:
tilt = pd.read_hdf(os.path.join(data_dir, "tilts.h5"))
tilt.index = pd.to_datetime(tilt.index, utc=True)

In [ ]:
df = flux.merge(pos, on=["Time tag", "C"]).merge(drivers, on="Time tag").merge(tilt, on="Time tag")

In [ ]:
df["cosMLTcosLAT"] = np.cos(df["MLT"]*np.pi/12) * np.cos(np.radians(df["LAT"]))
df["sinMLTcosLAT"] = np.sin(df["MLT"]*np.pi/12) * np.cos(np.radians(df["LAT"]))
df["sinLAT"] = np.sin(np.radians(df["LAT"]))

In [ ]:
df["lg E"] = np.log10(df["Energy"])

In [ ]:
best_features = [
    'bavg - max -0.25 0.0',
    'sme - current 0.0 0.0',
    'hp30 - current 0.0 0.0',
    'speed - mean -6.0 -4.0',
    'proton_density - max -0.25 0.0',
    'proton_density - mean -4.0 -3.0',
    'proton_density - current 0.0 0.0',
    'speed - current 0.0 0.0',
    'hp30 - mean -1.0 -0.75',
    'bavg - max -6.0 -4.0',
    'proton_density - mean -0.25 0.0',
    'sme - mean -0.25 0.0',
    'bz_gsm - fraction_negative -6.0 -4.0',
    'speed - mean -0.25 0.0',
    'sme - mean -1.5 -1.0',
    'speed - mean -24.0 -16.0',
    'by_gsm - fraction_negative -0.75 -0.5',
    'sme - mean -3.0 -2.0',
    'bz_gsm - fraction_negative -3.0 -2.0',
    'sme - mean -2.0 -1.5',
    'speed - mean -0.5 -0.25',
    'proton_density - max -0.75 -0.5',
    'sme - max -1.5 -1.0',
    'bz_gsm - fraction_negative -12.0 -8.0',
    'bavg - mean -24.0 -16.0',
    'proton_density - max -0.5 -0.25',
    'by_gsm - fraction_negative -3.0 -2.0',
    'bz_gsm - fraction_negative -4.0 -3.0',
    'bz_gsm - fraction_negative -1.5 -1.0',
    'proton_density - mean -0.5 -0.25',
    'bx_gsm - fraction_negative -0.25 0.0',
    'hp30 - max -24.0 -16.0',
    'bz_gsm - fraction_negative -0.75 -0.5',
    'bz_gsm - fraction_negative -0.5 -0.25',
    'sme - mean -0.5 -0.25',
    'bz_gsm - fraction_negative -2.0 -1.5'
    ]

In [ ]:
df = df[[*[f"Flux {a}" for a in np.arange(10, 180, 20)], "lg E", "RAD", "cosMLTcosLAT", "sinMLTcosLAT", "sinLAT", "Dipole tilt", *best_features]]

In [ ]:
# Drop nans
flux_cols = [col for col in df.columns if col.startswith("Flux ")]
non_flux_cols = df.columns.difference(flux_cols)
df = df.dropna(subset=non_flux_cols)

In [ ]:
x = df.iloc[:, 9:]
y = df.iloc[:, :9]
y = np.log10(1 + y)

In [ ]:
x

In [ ]:
two_week_bins = pd.to_datetime((x.index.view('int64') // (14 * 24 * 3600 * 10**9)) * (14 * 24 * 3600 * 10**9))
chunks = two_week_bins.unique()

rng = np.random.default_rng(0)

assignments = rng.choice(3, size=len(chunks), p=[0.7, 0.15, 0.15])

# Map each timestamp bin to its chunk number
chunk_ids = pd.factorize(two_week_bins)[0]

assigned = assignments[chunk_ids]

filter_train = assigned == 0
filter_test  = assigned == 1
filter_val   = assigned == 2

forced_start = pd.Timestamp('2015-03-10')
forced_end  = pd.Timestamp('2015-03-24')

forced_start = pd.to_datetime(forced_start, utc=True)
forced_end = pd.to_datetime(forced_end, utc=True)

forced_mask = (x.index >= forced_start) & (x.index < forced_end)

filter_test  = filter_test | forced_mask
filter_train = filter_train & ~forced_mask
filter_val   = filter_val & ~forced_mask

In [ ]:
x_test = x[filter_test]
y_test = y[filter_test]

x_val = x[filter_val]
y_val = y[filter_val]

x_train = x[filter_train]
y_train = y[filter_train]

In [ ]:
scaler = MinMaxScaler()
scaler.fit(x)

In [ ]:
joblib.dump(scaler, os.path.join(data_dir, "Datasets", "scaler.save"))

In [ ]:
stat, bins = np.histogram(df['sme - current 0.0 0.0'], bins=20)

idx = np.digitize(x_train['sme - current 0.0 0.0'].to_numpy(), bins) - 1
idx[idx == len(stat)] = len(stat) - 1
w_train_full = (stat[0] / stat[idx])**0.5
w_train_base = np.ones_like(w_train_full, dtype='float32')

idx = np.digitize(x_val['sme - current 0.0 0.0'].to_numpy(), bins) - 1
idx[idx == len(stat)] = len(stat) - 1
w_val_full = (stat[0] / stat[idx])**0.5
w_val_base = np.ones_like(w_val_full, dtype='float32')

idx = np.digitize(x_test['sme - current 0.0 0.0'].to_numpy(), bins) - 1
idx[idx == len(stat)] = len(stat) - 1
w_test_full = (stat[0] / stat[idx])**0.5
w_test_base = np.ones_like(w_test_full, dtype='float32')

In [ ]:
np.save(os.path.join(data_dir, "Datasets", "w_train.npy"), w_train_full)
np.save(os.path.join(data_dir, "Datasets", "w_test.npy"), w_test_full)
np.save(os.path.join(data_dir, "Datasets", "w_val.npy"), w_val_full)

In [ ]:
y_train.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets", "y_train.h5"), key='df', format='table')
y_test.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets", "y_test.h5"), key='df', format='table')
y_val.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets", "y_val.h5"), key='df', format='table')

In [ ]:
x_train[x_train.columns] = scaler.transform(x_train[x_train.columns])
x_test[x_test.columns] = scaler.transform(x_test[x_test.columns])
x_val[x_val.columns] = scaler.transform(x_val[x_val.columns])

x_train.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets", "x_train_scaled.h5"), key='df', format='table')
x_test.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets", "x_test_scaled.h5"), key='df', format='table')
x_val.astype(np.float32).to_hdf(os.path.join(data_dir, "Datasets", "x_val_scaled.h5"), key='df', format='table')